In [ ]:
!gdown --id 18UGswAy8rHpN73VWEmd3OhZGIUh-CvU-
!gdown --id 1nLPV8MXbSj4emxKvS8Rijl84jjz8ZgBI

/usr/local/lib/python3.10/dist-packages/gdown/cli.py:121: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=18UGswAy8rHpN73VWEmd3OhZGIUh-CvU-
To: /content/GPS 1.gpx
100% 16.0k/16.0k [00:00<00:00, 41.3MB/s]
/usr/local/lib/python3.10/dist-packages/gdown/cli.py:121: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1nLPV8MXbSj4emxKvS8Rijl84jjz8ZgBI
To: /content/GPS 2.gpx
100% 23.7k/23.7k [00:00<00:00, 70.2MB/s]


In [ ]:
pip install gpxpy

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gpxpy: filename=gpxpy-1.5.0-py3-none-any.whl size=42898 sha256=e15456729ac4ab9fa40f8a5a9a376334cf315ff059011b7c215ba8cb6779aeac
  Stored in directory: /root/.cache/pip/wheels/37/e1/72/25e1b018a67df6cb99583bef9b53b53e9ef59826ee514751aa
Successfully built gpxpy


In [ ]:
import gpxpy
gpx_file1 = open('GPS 1.gpx', 'r')
gpx_file2 = open('GPS 2.gpx', 'r')
gpx1 = gpxpy.parse(gpx_file1)
gpx2 = gpxpy.parse(gpx_file2)
len(gpx1.waypoints),len(gpx2.waypoints)

(156, 231)

In [ ]:
gps_arr1 = []
gps_arr2 = []
for data in gpx1.waypoints:
    gps_arr1.append([data.latitude, data.longitude, data.time])

for data in gpx2.waypoints:
    gps_arr2.append([data.latitude, data.longitude, data.time])

In [ ]:
import pandas as pd
gps1 = pd.DataFrame(gps_arr1, columns=['latitude', 'longitude', 'time'])
gps2 = pd.DataFrame(gps_arr2, columns=['latitude', 'longitude', 'time'])
gps1['suspect'] = 0
gps2['suspect'] = 1

In [ ]:
import folium
from folium.plugins import FastMarkerCluster

In [ ]:
def get_map(x):
    locations1 = list(zip(gps1[gps1.time.dt.hour==x]['latitude'].to_list(),gps1[gps1.time.dt.hour==x]['longitude'].to_list()))
    locations2 = list(zip(gps2[gps2.time.dt.hour==x]['latitude'].to_list(),gps2[gps2.time.dt.hour==x]['longitude'].to_list()))
    map1 = folium.Map(location=[52.3680, 4.9036], zoom_start=11.5)
    folium.PolyLine(locations1, color="red", weight=5.5, opacity=1).add_to(map1)
    for x in locations1:
        folium.Marker(
                location=x,
                popup="GPS 1 latitude:"+str(x[0])+" longitute:"+str(x[1]),
                icon=folium.Icon(color="red", icon="info-sign"),).add_to(map1)
    folium.PolyLine(locations2, color="green", weight=5.5, opacity=1).add_to(map1)
    for x in locations2:
        folium.Marker(
                location=x,
                popup="GPS 2 latitude:"+str(x[0])+" longitute:"+str(x[1]),
                icon=folium.Icon(color="green", icon="info-sign"),).add_to(map1)
    display(map1)

In [ ]:
get_map(15)

In [ ]:
get_map(16)

In [ ]:
df = pd.concat([gps1,gps2])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split( df[df.columns[:-2]],  df['suspect'], test_size=0.2)

In [ ]:
from xgboost import XGBClassifier


#inicialize XGBoost
model = XGBClassifier(max_depth=50, min_child_weight=1,  n_estimators=200,n_jobs=-1 , verbose=1,learning_rate=0.16)

# Add silent=True to avoid printing out updates with each cycle
model.fit(X_train, Y_train)

#make prediction
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(Y_test, y_pred))
print(classification_report(Y_test, y_pred))

[[23  7]
 [12 36]]
              precision    recall  f1-score   support

           0       0.66      0.77      0.71        30
           1       0.84      0.75      0.79        48

    accuracy                           0.76        78
   macro avg       0.75      0.76      0.75        78
weighted avg       0.77      0.76      0.76        78

